In [ ]:
import zipfile, io, os
from lxml import etree

EMU_INCH = 914400
def emu(inches): return str(int(inches * EMU_INCH))
def half_pt(pts): return str(int(pts * 100))
def emu_pt(pts): return str(int(pts * 12700))

NS = {
    'a':   'http://schemas.openxmlformats.org/drawingml/2006/main',
    'r':   'http://schemas.openxmlformats.org/officeDocument/2006/relationships',
    'p':   'http://schemas.openxmlformats.org/presentationml/2006/main',
    'rel': 'http://schemas.openxmlformats.org/package/2006/relationships',
    'ct':  'http://schemas.openxmlformats.org/package/2006/content-types',
}

def Q(prefix, local):
    return '{' + NS[prefix] + '}' + local

def rgb_hex(r,g,b):
    return f"{r:02X}{g:02X}{b:02X}"

TEAL   = rgb_hex(0x00,0xB4,0xD8)
DARK   = rgb_hex(0x1B,0x1B,0x1B)
ORANGE = rgb_hex(0xFF,0x6B,0x35)
GREEN  = rgb_hex(0x2E,0xCC,0x71)
WHITE  = rgb_hex(0xFF,0xFF,0xFF)
CREAM  = rgb_hex(0xF7,0xF4,0xEF)
LT_GRAY= rgb_hex(0xE8,0xE4,0xDE)
MD_GRAY= rgb_hex(0x6B,0x6B,0x7B)
RED    = rgb_hex(0xE7,0x4C,0x3C)
YELLOW = rgb_hex(0xF3,0x9C,0x12)
BLUE   = rgb_hex(0x34,0x98,0xDB)
SF_BLUE= rgb_hex(0x29,0xB5,0xE8)
CARD_BG= rgb_hex(0xF0,0xED,0xE7)
DIM    = rgb_hex(0x99,0x99,0xA8)
ACCENT_BG = rgb_hex(0xE6,0xF7,0xFB)

SW = emu(13.333)
SH = emu(7.5)
BG_COLOR = CREAM
TXT_COLOR = DARK

slides_xml = []

def make_sp_tree():
    tree = etree.SubElement(etree.Element('dummy'), Q('p','cSld'))
    bg = etree.SubElement(tree, Q('p','bg'))
    bgPr = etree.SubElement(bg, Q('p','bgPr'))
    sf = etree.SubElement(bgPr, Q('a','solidFill'))
    etree.SubElement(sf, Q('a','srgbClr'), val=BG_COLOR)
    etree.SubElement(bgPr, Q('a','effectLst'))
    sp_tree = etree.SubElement(tree, Q('p','spTree'))
    nvGrpSpPr = etree.SubElement(sp_tree, Q('p','nvGrpSpPr'))
    cNvPr = etree.SubElement(nvGrpSpPr, Q('p','cNvPr'), id="1", name="")
    etree.SubElement(nvGrpSpPr, Q('p','cNvGrpSpPr'))
    etree.SubElement(nvGrpSpPr, Q('p','nvPr'))
    grpSpPr = etree.SubElement(sp_tree, Q('p','grpSpPr'))
    xfrm = etree.SubElement(grpSpPr, Q('a','xfrm'))
    etree.SubElement(xfrm, Q('a','off'), x="0", y="0")
    etree.SubElement(xfrm, Q('a','ext'), cx=SW, cy=SH)
    etree.SubElement(xfrm, Q('a','chOff'), x="0", y="0")
    etree.SubElement(xfrm, Q('a','chExt'), cx=SW, cy=SH)
    return tree, sp_tree

def add_rect_shape(sp_tree, x, y, w, h, fill_color, shape_id=2):
    sp = etree.SubElement(sp_tree, Q('p','sp'))
    nvSpPr = etree.SubElement(sp, Q('p','nvSpPr'))
    etree.SubElement(nvSpPr, Q('p','cNvPr'), id=str(shape_id), name="rect")
    etree.SubElement(nvSpPr, Q('p','cNvSpPr'))
    etree.SubElement(nvSpPr, Q('p','nvPr'))
    spPr = etree.SubElement(sp, Q('p','spPr'))
    xfrm = etree.SubElement(spPr, Q('a','xfrm'))
    etree.SubElement(xfrm, Q('a','off'), x=emu(x), y=emu(y))
    etree.SubElement(xfrm, Q('a','ext'), cx=emu(w), cy=emu(h))
    etree.SubElement(spPr, Q('a','prstGeom'), prst="rect")
    sf = etree.SubElement(spPr, Q('a','solidFill'))
    etree.SubElement(sf, Q('a','srgbClr'), val=fill_color)
    ln = etree.SubElement(spPr, Q('a','ln'))
    etree.SubElement(ln, Q('a','noFill'))
    return sp

def add_rounded_rect(sp_tree, x, y, w, h, fill_color, border_color=None, shape_id=2):
    sp = etree.SubElement(sp_tree, Q('p','sp'))
    nvSpPr = etree.SubElement(sp, Q('p','nvSpPr'))
    etree.SubElement(nvSpPr, Q('p','cNvPr'), id=str(shape_id), name="rrect")
    etree.SubElement(nvSpPr, Q('p','cNvSpPr'))
    etree.SubElement(nvSpPr, Q('p','nvPr'))
    spPr = etree.SubElement(sp, Q('p','spPr'))
    xfrm = etree.SubElement(spPr, Q('a','xfrm'))
    etree.SubElement(xfrm, Q('a','off'), x=emu(x), y=emu(y))
    etree.SubElement(xfrm, Q('a','ext'), cx=emu(w), cy=emu(h))
    etree.SubElement(spPr, Q('a','prstGeom'), prst="roundRect")
    sf = etree.SubElement(spPr, Q('a','solidFill'))
    etree.SubElement(sf, Q('a','srgbClr'), val=fill_color)
    ln = etree.SubElement(spPr, Q('a','ln'))
    if border_color:
        bsf = etree.SubElement(ln, Q('a','solidFill'))
        etree.SubElement(bsf, Q('a','srgbClr'), val=border_color)
    else:
        etree.SubElement(ln, Q('a','noFill'))
    return sp

def add_oval(sp_tree, x, y, size, fill_color, text=None, text_size=22, text_color=WHITE, shape_id=2):
    sp = etree.SubElement(sp_tree, Q('p','sp'))
    nvSpPr = etree.SubElement(sp, Q('p','nvSpPr'))
    etree.SubElement(nvSpPr, Q('p','cNvPr'), id=str(shape_id), name="oval")
    etree.SubElement(nvSpPr, Q('p','cNvSpPr'))
    etree.SubElement(nvSpPr, Q('p','nvPr'))
    spPr = etree.SubElement(sp, Q('p','spPr'))
    xfrm = etree.SubElement(spPr, Q('a','xfrm'))
    etree.SubElement(xfrm, Q('a','off'), x=emu(x), y=emu(y))
    etree.SubElement(xfrm, Q('a','ext'), cx=emu(size), cy=emu(size))
    etree.SubElement(spPr, Q('a','prstGeom'), prst="ellipse")
    sf = etree.SubElement(spPr, Q('a','solidFill'))
    etree.SubElement(sf, Q('a','srgbClr'), val=fill_color)
    ln = etree.SubElement(spPr, Q('a','ln'))
    etree.SubElement(ln, Q('a','noFill'))
    if text:
        txBody = etree.SubElement(sp, Q('p','txBody'))
        bodyPr = etree.SubElement(txBody, Q('a','bodyPr'), anchor="ctr", wrap="none")
        etree.SubElement(txBody, Q('a','lstStyle'))
        p = etree.SubElement(txBody, Q('a','p'))
        pPr = etree.SubElement(p, Q('a','pPr'), algn="ctr")
        r = etree.SubElement(p, Q('a','r'))
        rPr = etree.SubElement(r, Q('a','rPr'), lang="en-US", sz=half_pt(text_size), b="1")
        sf2 = etree.SubElement(rPr, Q('a','solidFill'))
        etree.SubElement(sf2, Q('a','srgbClr'), val=text_color)
        etree.SubElement(rPr, Q('a','latin'), typeface="Calibri")
        etree.SubElement(r, Q('a','t')).text = text
    return sp

def make_txbox(sp_tree, x, y, w, h, paragraphs, shape_id=2):
    sp = etree.SubElement(sp_tree, Q('p','sp'))
    nvSpPr = etree.SubElement(sp, Q('p','nvSpPr'))
    etree.SubElement(nvSpPr, Q('p','cNvPr'), id=str(shape_id), name="txBox")
    cNvSpPr = etree.SubElement(nvSpPr, Q('p','cNvSpPr'), txBox="1")
    etree.SubElement(nvSpPr, Q('p','nvPr'))
    spPr = etree.SubElement(sp, Q('p','spPr'))
    xfrm = etree.SubElement(spPr, Q('a','xfrm'))
    etree.SubElement(xfrm, Q('a','off'), x=emu(x), y=emu(y))
    etree.SubElement(xfrm, Q('a','ext'), cx=emu(w), cy=emu(h))
    etree.SubElement(spPr, Q('a','prstGeom'), prst="rect")
    etree.SubElement(spPr, Q('a','noFill'))
    ln = etree.SubElement(spPr, Q('a','ln'))
    etree.SubElement(ln, Q('a','noFill'))
    txBody = etree.SubElement(sp, Q('p','txBody'))
    bodyPr = etree.SubElement(txBody, Q('a','bodyPr'), wrap="square", rtlCol="0")
    etree.SubElement(txBody, Q('a','lstStyle'))
    for i, (text, font_size, color, bold, align) in enumerate(paragraphs):
        p = etree.SubElement(txBody, Q('a','p'))
        pPr = etree.SubElement(p, Q('a','pPr'), algn=align)
        if i > 0:
            spcBef = etree.SubElement(pPr, Q('a','spcBef'))
            etree.SubElement(spcBef, Q('a','spcPts'), val=str(int(font_size*40)))
        r = etree.SubElement(p, Q('a','r'))
        rPr_attrs = {'lang': 'en-US', 'sz': half_pt(font_size), 'dirty': '0'}
        if bold:
            rPr_attrs['b'] = '1'
        rPr = etree.SubElement(r, Q('a','rPr'), **rPr_attrs)
        sf = etree.SubElement(rPr, Q('a','solidFill'))
        etree.SubElement(sf, Q('a','srgbClr'), val=color)
        etree.SubElement(rPr, Q('a','latin'), typeface="Calibri")
        etree.SubElement(r, Q('a','t')).text = text
    return sp

def T(text, size, color, bold=False, align='l'):
    return (text, size, color, bold, align)

shape_counter = [1]
def sid():
    shape_counter[0] += 1
    return shape_counter[0]

print("Light theme PPTX builder ready")

In [ ]:
slides_data = []

# === SLIDE 1: TITLE ===
def build_slide_1():
    cSld, sp = make_sp_tree()
    add_rect_shape(sp, 0, 0, 13.333, 0.06, TEAL, sid())
    add_rect_shape(sp, 0, 7.44, 13.333, 0.06, ORANGE, sid())
    make_txbox(sp, 0.8, 1.4, 11.7, 0.7, [T("CoCoDuo Migration Launchpad", 42, DARK, True, 'ctr')], sid())
    make_txbox(sp, 0.8, 2.2, 11.7, 0.5, [T("AI-Powered Migration Planning for Snowflake", 24, TEAL, False, 'ctr')], sid())
    add_rect_shape(sp, 5.5, 3.1, 2.3, 0.04, ORANGE, sid())
    make_txbox(sp, 0.8, 3.6, 11.7, 0.4, [T("How We Built an Intelligent Migration Assessment Engine in 6 Hours", 18, MD_GRAY, False, 'ctr')], sid())
    make_txbox(sp, 0.8, 4.2, 11.7, 0.4, [T("Using Snowflake Cortex, Notebooks, and Agentic AI", 16, MD_GRAY, False, 'ctr')], sid())
    make_txbox(sp, 0.8, 5.6, 11.7, 0.4, [T("kipi.ai  |  Coconut Duo Team  |  March 2026", 14, DIM, False, 'ctr')], sid())
    return cSld

# === SLIDE 2: PROBLEM ===
def build_slide_2():
    cSld, sp = make_sp_tree()
    add_rect_shape(sp, 0, 0, 13.333, 0.06, TEAL, sid())
    make_txbox(sp, 0.8, 0.3, 11.7, 0.5, [T("The Problem: Migration Planning Is Broken", 30, DARK, True, 'l')], sid())
    add_rect_shape(sp, 0.8, 0.95, 2.5, 0.04, ORANGE, sid())
    probs = [
        (RED,    "Weeks of Manual Effort",      "Migration assessments take 2-4 weeks of SA/consultant time per customer. Scoping calls, spreadsheets, back-and-forth emails, manual document assembly."),
        (ORANGE, "No Standardized Framework",    "Every SA builds plans differently. No consistent methodology, no reusable templates, no institutional knowledge captured."),
        (YELLOW, "Tool Selection Paralysis",     "Customers face 50+ tools across ingestion, transformation, orchestration, BI, governance. SAs spend hours researching."),
        (BLUE,   "Inconsistent Quality",         "Plan quality varies wildly by SA experience. Junior SAs struggle with complex multi-source migrations. No guardrails."),
    ]
    for i,(c,title,desc) in enumerate(probs):
        y = 1.4 + i*1.35
        add_rect_shape(sp, 0.8, y, 0.08, 1.05, c, sid())
        make_txbox(sp, 1.15, y+0.05, 5.0, 0.35, [T(title, 16, DARK, True, 'l')], sid())
        make_txbox(sp, 1.15, y+0.4, 5.5, 0.65, [T(desc, 11.5, MD_GRAY, False, 'l')], sid())
    add_rounded_rect(sp, 7.2, 1.4, 5.3, 5.2, WHITE, TEAL, sid())
    make_txbox(sp, 7.5, 1.6, 4.7, 0.4, [T("Real Impact \u2014 Horizon Residential Case", 15, TEAL, True, 'l')], sid())
    case_items = [
        "\u25B8  5 source systems (SQL Server, RealPage, Funnel, SuiteSpot, PeopleSoft)",
        "\u25B8  12 TB total data across on-prem & SaaS sources",
        "\u25B8  Undocumented SSIS packages & SSAS Cubes",
        "\u25B8  Legacy SSRS dashboards used daily by executives",
        "\u25B8  Power BI workflows with embedded business logic",
        "\u25B8  Operations team crashing databases \u2014 no access control",
    ]
    paras = [(item, 12, DARK, False, 'l') for item in case_items]
    paras.append(("", 8, DARK, False, 'l'))
    paras.append(("Without CoCo: 3-4 weeks of manual planning", 13, RED, True, 'l'))
    paras.append(("With CoCo: Comprehensive plan in ~30 minutes", 13, GREEN, True, 'l'))
    make_txbox(sp, 7.5, 2.15, 4.7, 4.2, paras, sid())
    return cSld

# === SLIDE 3: SOLUTION ===
def build_slide_3():
    cSld, sp = make_sp_tree()
    add_rect_shape(sp, 0, 0, 13.333, 0.06, TEAL, sid())
    make_txbox(sp, 0.8, 0.3, 11.7, 0.5, [T("The CoCo Solution: AI-Powered Migration Launchpad", 30, DARK, True, 'l')], sid())
    add_rect_shape(sp, 0.8, 0.95, 2.5, 0.04, ORANGE, sid())
    make_txbox(sp, 0.8, 1.2, 11.7, 0.4, [T("An agentic AI system built entirely on Snowflake that transforms scoping notes into production-ready migration plans.", 14, MD_GRAY, False, 'l')], sid())
    steps = [
        ("1", "Intake",           "SA enters scoping notes\n(free-form text from calls)", TEAL),
        ("2", "AI Analysis",      "Cortex AI agent parses\nnotes, maps to schema", BLUE),
        ("3", "Knowledge Base",   "Matches tool profiles,\nplaybooks, ref architectures", ORANGE),
        ("4", "Plan Generation",  "72+ KB migration plan:\nphases, tools, risks, SLAs", GREEN),
        ("5", "Branded Report",   "kipi.ai branded HTML\nreport for customer delivery", rgb_hex(0x9B,0x59,0xB6)),
    ]
    for i,(num,title,desc,color) in enumerate(steps):
        x = 0.5 + i*2.5
        add_oval(sp, x+0.65, 1.9, 0.6, color, num, 22, WHITE, sid())
        make_txbox(sp, x, 2.65, 2.2, 0.35, [T(title, 14, DARK, True, 'ctr')], sid())
        for j,line in enumerate(desc.split('\n')):
            make_txbox(sp, x, 3.0+j*0.28, 2.2, 0.3, [T(line, 11, MD_GRAY, False, 'ctr')], sid())
        if i < 4:
            make_txbox(sp, x+2.05, 1.95, 0.5, 0.5, [T("\u27A4", 20, DIM, False, 'ctr')], sid())
    components = [
        ("Migration Expert Skill", "40-page knowledge base: architecture patterns, source-system playbooks, tool selection matrix, multi-tenancy patterns, brand guidelines"),
        ("Backend Data Model", "4 Snowflake tables: Assessments, Source Systems, ETL Tools, Reference Architectures \u2014 structured intake schema"),
        ("Plan Renderer Notebook", "Snowflake Notebook reads JSON plan, enriches with tool DB, generates branded 72 KB HTML with 14 sections"),
        ("Cortex Code Agent", "Agentic AI orchestrator \u2014 parses free-form notes, queries knowledge base, generates plan, loads data, renders report"),
    ]
    for i,(title,desc) in enumerate(components):
        col = i % 2
        row = i // 2
        x = 0.8 + col*6.1
        y = 4.1 + row*1.35
        add_rounded_rect(sp, x, y, 5.7, 1.15, WHITE, LT_GRAY, sid())
        make_txbox(sp, x+0.2, y+0.1, 5.3, 0.3, [T(title, 13, TEAL, True, 'l')], sid())
        make_txbox(sp, x+0.2, y+0.45, 5.3, 0.6, [T(desc, 10.5, MD_GRAY, False, 'l')], sid())
    return cSld

# === SLIDE 4: OLD vs NEW ===
def build_slide_4():
    cSld, sp = make_sp_tree()
    add_rect_shape(sp, 0, 0, 13.333, 0.06, TEAL, sid())
    make_txbox(sp, 0.8, 0.3, 11.7, 0.5, [T("Old Way vs. CoCo Way", 30, DARK, True, 'l')], sid())
    add_rect_shape(sp, 0.8, 0.95, 2.5, 0.04, ORANGE, sid())
    add_rect_shape(sp, 0.5, 1.3, 5.9, 0.5, rgb_hex(0xFC,0xE4,0xE4), sid())
    make_txbox(sp, 0.5, 1.32, 5.9, 0.45, [T("Traditional Approach", 18, RED, True, 'ctr')], sid())
    add_rect_shape(sp, 6.9, 1.3, 5.9, 0.5, rgb_hex(0xE4,0xFC,0xE9), sid())
    make_txbox(sp, 6.9, 1.32, 5.9, 0.45, [T("CoCo Approach", 18, GREEN, True, 'ctr')], sid())
    add_rect_shape(sp, 6.4, 1.3, 0.08, 5.7, LT_GRAY, sid())
    old = [
        "\u2717  Scoping call \u2192 handwritten notes \u2192 spreadsheet",
        "\u2717  SA manually researches 50+ tools per engagement",
        "\u2717  Architecture designed from scratch every time",
        "\u2717  2-4 weeks per customer assessment",
        "\u2717  No institutional knowledge capture",
        "\u2717  Plan quality depends on individual SA experience",
        "\u2717  Word/PPT docs \u2014 no structured data, not reusable",
        "\u2717  No reference architecture matching",
    ]
    new = [
        "\u2713  Scoping notes \u2192 AI agent \u2192 structured assessment",
        "\u2713  Tool Selection Matrix auto-matched to customer needs",
        "\u2713  Medallion, Hub-Spoke patterns auto-selected",
        "\u2713  ~30 minutes: intake to branded deliverable",
        "\u2713  Every plan enriches the knowledge base for future",
        "\u2713  Consistent quality with best-practice guardrails",
        "\u2713  Structured JSON in Snowflake + branded HTML report",
        "\u2713  AI matches similar past migrations automatically",
    ]
    for i,text in enumerate(old):
        make_txbox(sp, 0.7, 2.0+i*0.6, 5.5, 0.5, [T(text, 12.5, rgb_hex(0xAA,0x33,0x33), False, 'l')], sid())
    for i,text in enumerate(new):
        make_txbox(sp, 7.1, 2.0+i*0.6, 5.5, 0.5, [T(text, 12.5, rgb_hex(0x1A,0x8A,0x3A), False, 'l')], sid())
    return cSld

slides_data = [build_slide_1(), build_slide_2(), build_slide_3(), build_slide_4()]
print(f"Built {len(slides_data)} slides (light theme)")

In [ ]:
# === SLIDE 5: TIME SAVED ===
def build_slide_5():
    cSld, sp = make_sp_tree()
    add_rect_shape(sp, 0, 0, 13.333, 0.06, TEAL, sid())
    make_txbox(sp, 0.8, 0.3, 11.7, 0.5, [T("Time Savings: From Weeks to Minutes", 30, DARK, True, 'l')], sid())
    add_rect_shape(sp, 0.8, 0.95, 2.5, 0.04, ORANGE, sid())
    add_rect_shape(sp, 0.5, 1.4, 12.3, 0.45, TEAL, sid())
    headers = [("Phase",0.6,2.4),("Old Duration",3.1,1.5),("CoCo Duration",4.7,1.5),("Savings",6.3,0.9),("How CoCo Does It",7.4,5.2)]
    for txt,x,w in headers:
        make_txbox(sp, x, 1.42, w, 0.4, [T(txt, 11, WHITE, True, 'ctr')], sid())
    rows = [
        ("Scoping & Intake",       "2-3 days", "~5 min",  "97%", "AI parses free-form scoping notes, extracts entities, maps to schema"),
        ("Source System Analysis",  "3-5 days", "~5 min",  "98%", "Knowledge base auto-matches source system playbooks"),
        ("Tool Selection",          "2-3 days", "~3 min",  "97%", "Tool Selection Matrix (50+ tools) auto-compared for customer context"),
        ("Architecture Design",     "3-5 days", "~5 min",  "97%", "Patterns auto-selected; warehouses, schemas, RBAC designed"),
        ("Risk & SLA Analysis",     "1-2 days", "~2 min",  "96%", "AI identifies risks from notes, maps to mitigation patterns"),
        ("Report Generation",       "2-3 days", "~10 min", "95%", "72 KB branded HTML with 14 sections, auto-rendered from JSON"),
    ]
    row_bg1 = WHITE
    row_bg2 = rgb_hex(0xF2,0xEF,0xE9)
    for i,(phase,old,new,sav,how) in enumerate(rows):
        y = 2.05 + i*0.72
        bg = row_bg1 if i%2==0 else row_bg2
        add_rect_shape(sp, 0.5, y, 12.3, 0.6, bg, sid())
        make_txbox(sp, 0.6, y+0.1, 2.4, 0.4, [T(phase, 11.5, DARK, True, 'l')], sid())
        make_txbox(sp, 3.1, y+0.1, 1.5, 0.4, [T(old, 11.5, RED, False, 'ctr')], sid())
        make_txbox(sp, 4.7, y+0.1, 1.5, 0.4, [T(new, 11.5, GREEN, True, 'ctr')], sid())
        make_txbox(sp, 6.3, y+0.1, 0.9, 0.4, [T(sav, 12, TEAL, True, 'ctr')], sid())
        make_txbox(sp, 7.4, y+0.1, 5.2, 0.4, [T(how, 10, MD_GRAY, False, 'l')], sid())
    ty = 6.45
    add_rect_shape(sp, 0.5, ty, 12.3, 0.55, rgb_hex(0xE4,0xFC,0xE9), sid())
    make_txbox(sp, 0.6, ty+0.08, 2.4, 0.4, [T("TOTAL", 14, GREEN, True, 'l')], sid())
    make_txbox(sp, 3.1, ty+0.08, 1.5, 0.4, [T("2-4 weeks", 13, RED, True, 'ctr')], sid())
    make_txbox(sp, 4.7, ty+0.08, 1.5, 0.4, [T("~30 min", 13, GREEN, True, 'ctr')], sid())
    make_txbox(sp, 6.3, ty+0.08, 0.9, 0.4, [T("~99%", 14, TEAL, True, 'ctr')], sid())
    make_txbox(sp, 7.4, ty+0.08, 5.2, 0.4, [T("End-to-end: scoping notes in \u2192 branded migration plan out", 11, DARK, True, 'l')], sid())
    return cSld

# === SLIDE 6: OUTCOMES ===
def build_slide_6():
    cSld, sp = make_sp_tree()
    add_rect_shape(sp, 0, 0, 13.333, 0.06, TEAL, sid())
    make_txbox(sp, 0.8, 0.3, 11.7, 0.5, [T("Expected Outcomes & Measurable Impact", 30, DARK, True, 'l')], sid())
    add_rect_shape(sp, 0.8, 0.95, 2.5, 0.04, ORANGE, sid())
    metrics = [
        ("99%",  "Time Reduction", "2-4 weeks \u2192 30 min\nper assessment", GREEN),
        ("14",   "Plan Sections",  "Exec Summary, Architecture,\nTools, Phases, Risks, SLAs", TEAL),
        ("50+",  "Tools Evaluated", "Auto-compared from curated\nselection matrix", BLUE),
        ("5",    "Customers Loaded","Knowledge base growing\nwith every engagement", ORANGE),
    ]
    for i,(num,label,desc,color) in enumerate(metrics):
        x = 0.5 + i*3.15
        add_rounded_rect(sp, x, 1.3, 2.9, 1.8, WHITE, LT_GRAY, sid())
        make_txbox(sp, x, 1.4, 2.9, 0.55, [T(num, 36, color, True, 'ctr')], sid())
        make_txbox(sp, x, 2.0, 2.9, 0.3, [T(label, 13, DARK, True, 'ctr')], sid())
        for j,line in enumerate(desc.split('\n')):
            make_txbox(sp, x, 2.35+j*0.28, 2.9, 0.3, [T(line, 10.5, MD_GRAY, False, 'ctr')], sid())
    outcomes = [
        ("For Solutions Architects", [
            "\u25B8  Focus on relationships, not document assembly",
            "\u25B8  Consistent quality regardless of experience",
            "\u25B8  Built-in best practices from past engagements",
        ], TEAL),
        ("For Customers", [
            "\u25B8  Faster time-to-value: plan within first meeting",
            "\u25B8  Transparent tool comparisons with pros/cons",
            "\u25B8  Clear phased roadmap with risks identified upfront",
        ], GREEN),
        ("For kipi.ai", [
            "\u25B8  Scale 10x more prospects without headcount",
            "\u25B8  Institutional knowledge compounding over time",
            "\u25B8  Competitive differentiator: no other SI has this",
        ], ORANGE),
    ]
    for i,(title,items,color) in enumerate(outcomes):
        x = 0.5 + i*4.2
        add_rounded_rect(sp, x, 3.6, 3.9, 3.4, WHITE, color, sid())
        make_txbox(sp, x+0.2, 3.75, 3.5, 0.35, [T(title, 15, color, True, 'l')], sid())
        paras = [(item, 12, DARK, False, 'l') for item in items]
        make_txbox(sp, x+0.2, 4.3, 3.5, 2.5, paras, sid())
    return cSld

# === SLIDE 7: TECH STACK ===
def build_slide_7():
    cSld, sp = make_sp_tree()
    add_rect_shape(sp, 0, 0, 13.333, 0.06, TEAL, sid())
    make_txbox(sp, 0.8, 0.3, 11.7, 0.5, [T("Technology Stack & Architecture", 30, DARK, True, 'l')], sid())
    add_rect_shape(sp, 0.8, 0.95, 2.5, 0.04, ORANGE, sid())
    make_txbox(sp, 0.8, 1.15, 11.7, 0.35, [T("100% built on Snowflake \u2014 no external infrastructure, no additional cloud services, no API keys.", 13, MD_GRAY, False, 'l')], sid())
    layers = [
        ("AI & Intelligence Layer", [
            ("Snowflake Cortex Code", "Agentic AI orchestrator"),
            ("Cortex AI Skills", "40-page knowledge base"),
            ("Claude Opus 4 (via Cortex)", "Foundation model for reasoning"),
        ], TEAL),
        ("Compute & Processing", [
            ("Snowflake Notebooks", "JSON \u2192 branded HTML report"),
            ("Notebook Services (CPU)", "Jinja2 / lxml rendering"),
            ("Snowflake Warehouses", "Data ops and queries"),
        ], BLUE),
        ("Data & Storage Layer", [
            ("Snowflake Tables (4)", "Assessments, Sources, ETL, Refs"),
            ("Internal Stages", "@MIGRATION_PLANS for HTML"),
            ("GET_PRESIGNED_URL", "Secure download links"),
        ], GREEN),
    ]
    for i,(layer,items,color) in enumerate(layers):
        y = 1.7 + i*1.85
        add_rect_shape(sp, 0.5, y, 0.1, 1.55, color, sid())
        make_txbox(sp, 0.8, y, 3.0, 0.35, [T(layer, 14, color, True, 'l')], sid())
        for j,(tool,desc) in enumerate(items):
            x = 0.8 + j*4.1
            add_rounded_rect(sp, x, y+0.42, 3.8, 0.95, WHITE, LT_GRAY, sid())
            make_txbox(sp, x+0.15, y+0.48, 3.5, 0.25, [T(tool, 11.5, DARK, True, 'l')], sid())
            make_txbox(sp, x+0.15, y+0.75, 3.5, 0.5, [T(desc, 10, MD_GRAY, False, 'l')], sid())
    make_txbox(sp, 0.8, 7.0, 11.7, 0.3, [T("Key Libraries: python-pptx | Jinja2 | JSON | lxml", 11, DIM, False, 'ctr')], sid())
    return cSld

# === SLIDE 8: TEAM ROLES ===
def build_slide_8():
    cSld, sp = make_sp_tree()
    add_rect_shape(sp, 0, 0, 13.333, 0.06, TEAL, sid())
    make_txbox(sp, 0.8, 0.3, 11.7, 0.5, [T("Team Roles & Responsibilities", 30, DARK, True, 'l')], sid())
    add_rect_shape(sp, 0.8, 0.95, 2.5, 0.04, ORANGE, sid())
    roles = [
        ("SA", "SA / Consultant", "Primary User", [
            "Conducts customer scoping calls",
            "Feeds scoping notes into CoCo",
            "Reviews & refines generated plan",
            "Presents deliverable to customer",
        ], TEAL),
        ("CO", "CoCo AI Agent", "AI Engine", [
            "Parses free-form scoping notes",
            "Maps to structured schema",
            "Matches tools, patterns, refs",
            "Generates migration plan",
        ], ORANGE),
        ("KB", "Knowledge Base", "Institutional Memory", [
            "40-page migration expertise",
            "50+ tool profiles w/ pricing",
            "Architecture patterns & playbooks",
            "Reference architectures",
        ], GREEN),
        ("SF", "Snowflake Platform", "Infrastructure", [
            "Stores assessment data (4 tables)",
            "Runs Notebook for rendering",
            "Hosts HTML on internal stage",
            "Presigned URLs for download",
        ], SF_BLUE),
    ]
    for i,(initials,role,subtitle,items,color) in enumerate(roles):
        x = 0.4 + i*3.2
        add_rounded_rect(sp, x, 1.3, 3.0, 5.7, WHITE, color, sid())
        add_oval(sp, x+0.85, 1.55, 1.1, color, initials, 24, WHITE, sid())
        make_txbox(sp, x+0.1, 2.85, 2.8, 0.35, [T(role, 14, DARK, True, 'ctr')], sid())
        make_txbox(sp, x+0.1, 3.2, 2.8, 0.25, [T(subtitle, 11, color, False, 'ctr')], sid())
        for j,item in enumerate(items):
            make_txbox(sp, x+0.2, 3.7+j*0.6, 2.6, 0.5, [T("\u25B8  " + item, 11, MD_GRAY, False, 'l')], sid())
    return cSld

slides_data.extend([build_slide_5(), build_slide_6(), build_slide_7(), build_slide_8()])
print(f"Built {len(slides_data)} slides total (light theme)")

In [ ]:
# === SLIDE 9: ASSUMPTIONS, RISKS, WORKAROUNDS ===
def build_slide_9():
    cSld, sp = make_sp_tree()
    add_rect_shape(sp, 0, 0, 13.333, 0.06, TEAL, sid())
    make_txbox(sp, 0.8, 0.3, 11.7, 0.5, [T("Assumptions, Risks & Trial Account Workarounds", 28, DARK, True, 'l')], sid())
    add_rect_shape(sp, 0.8, 0.95, 2.5, 0.04, ORANGE, sid())
    sections = [
        (0.5,  3.9, "Assumptions", YELLOW, DARK, [
            "SA has conducted at least one scoping call",
            "Free-form notes capture enough detail for AI",
            "Tool Selection Matrix kept current (~quarterly)",
            "Ref architectures added after each engagement",
            "Snowflake Enterprise edition for production",
        ]),
        (4.7,  3.9, "Risks & Limitations", RED, WHITE, [
            "AI may hallucinate tool details \u2014 human review required",
            "Complex notes may need iterative refinement with AI",
            "Knowledge base requires ongoing curation",
            "Trial account has limited credits & models",
            "No Snowflake Data Sharing in trial \u2014 used mock data",
        ]),
        (8.9,  3.9, "Open Questions", TEAL, WHITE, [
            "Customer-facing plans or internal-only?",
            "CRM integration (Salesforce/HubSpot) for auto-intake?",
            "Multi-language support for intl. prospects?",
            "How to handle sensitive info in ref architectures?",
            "Pricing model: per-assessment or platform fee?",
        ]),
    ]
    for sx, sw, title, hdr_color, hdr_txt_color, items in sections:
        add_rect_shape(sp, sx, 1.25, sw, 0.4, hdr_color, sid())
        make_txbox(sp, sx, 1.27, sw, 0.35, [T(title, 13, hdr_txt_color, True, 'ctr')], sid())
        for i, text in enumerate(items):
            make_txbox(sp, sx+0.1, 1.78+i*0.42, sw-0.2, 0.4, [T("\u25B8  "+text, 10, MD_GRAY, False, 'l')], sid())
    add_rect_shape(sp, 0.5, 4.15, 12.3, 0.04, LT_GRAY, sid())
    make_txbox(sp, 0.8, 4.35, 11.7, 0.35, [T("Trial Account Workarounds", 17, ORANGE, True, 'l')], sid())
    tbl = [
        ("Limitation",                     "Workaround",                                                                  "Impact",                                                True),
        ("No Snowflake Data Sharing",      "Simulated with staged JSON data and Snowpipe patterns",                       "Same architecture; swap for real Shares in prod",      False),
        ("Limited Cortex Models",           "Used Claude Opus 4 via Cortex Code (available in trial)",                    "No impact \u2014 same workflow, different model",             False),
        ("No Network Policies",             "Demonstrated RBAC via roles/grants instead",                                 "Network policies additive in production",              False),
        ("30-day Trial Expiry",             "All artifacts exportable and version-controlled",                            "Full portability to any Snowflake account",            False),
        ("No External Stages (S3/Azure)",  "Used Snowflake Internal Stages for all file ops",                             "Swap to external stages for cloud-native ingestion",   False),
    ]
    for i,(c1,c2,c3,is_hdr) in enumerate(tbl):
        y = 4.85 + i*0.38
        bg = TEAL if is_hdr else (WHITE if i%2==0 else rgb_hex(0xF2,0xEF,0xE9))
        tc = WHITE if is_hdr else MD_GRAY
        add_rect_shape(sp, 0.5, y, 12.3, 0.36, bg, sid())
        make_txbox(sp, 0.6, y+0.03, 3.0, 0.3, [T(c1, 9.5, tc, is_hdr, 'l')], sid())
        make_txbox(sp, 3.7, y+0.03, 5.3, 0.3, [T(c2, 9.5, tc, is_hdr, 'l')], sid())
        make_txbox(sp, 9.2, y+0.03, 3.4, 0.3, [T(c3, 9.5, tc, is_hdr, 'l')], sid())
    return cSld

# === SLIDE 10: CLOSING ===
def build_slide_10():
    cSld, sp = make_sp_tree()
    add_rect_shape(sp, 0, 0, 13.333, 0.06, TEAL, sid())
    add_rect_shape(sp, 0, 7.44, 13.333, 0.06, ORANGE, sid())
    make_txbox(sp, 0.8, 1.6, 11.7, 0.7, [T("CoCoDuo Migration Launchpad", 40, DARK, True, 'ctr')], sid())
    make_txbox(sp, 0.8, 2.4, 11.7, 0.5, [T("Built in 6 Hours. Saves Weeks. Scales Infinitely.", 22, TEAL, False, 'ctr')], sid())
    add_rect_shape(sp, 5.5, 3.2, 2.3, 0.04, ORANGE, sid())
    stats = [("6 hrs","Build Time"),("~30 min","Per Plan"),("99%","Time Saved"),("14","Plan Sections"),("50+","Tools Covered")]
    for i,(val,label) in enumerate(stats):
        x = 0.8 + i*2.45
        add_rounded_rect(sp, x, 3.7, 2.15, 1.2, WHITE, LT_GRAY, sid())
        make_txbox(sp, x, 3.8, 2.15, 0.5, [T(val, 26, TEAL, True, 'ctr')], sid())
        make_txbox(sp, x, 4.4, 2.15, 0.35, [T(label, 12, MD_GRAY, False, 'ctr')], sid())
    make_txbox(sp, 0.8, 5.5, 11.7, 0.4, [T('"There is no AI Strategy without a Data Strategy."', 18, ORANGE, True, 'ctr')], sid())
    make_txbox(sp, 0.8, 6.2, 11.7, 0.4, [T("Coconut Duo Team  |  kipi.ai  |  March 2026", 14, DIM, False, 'ctr')], sid())
    return cSld

slides_data.extend([build_slide_9(), build_slide_10()])
print(f"All {len(slides_data)} slides built (light theme)")

# ============================================================
# ASSEMBLE PPTX ZIP
# ============================================================
def make_pptx(slides, output_path):
    decl = b'<?xml version="1.0" encoding="UTF-8" standalone="yes"?>\r\n'
    n = len(slides)
    content_types = etree.Element(Q('ct','Types'))
    etree.SubElement(content_types, Q('ct','Default'), Extension="rels", ContentType="application/vnd.openxmlformats-package.relationships+xml")
    etree.SubElement(content_types, Q('ct','Default'), Extension="xml", ContentType="application/xml")
    etree.SubElement(content_types, Q('ct','Override'), PartName="/ppt/presentation.xml", ContentType="application/vnd.openxmlformats-officedocument.presentationml.presentation.main+xml")
    for i in range(n):
        etree.SubElement(content_types, Q('ct','Override'), PartName=f"/ppt/slides/slide{i+1}.xml", ContentType="application/vnd.openxmlformats-officedocument.presentationml.slide+xml")
    etree.SubElement(content_types, Q('ct','Override'), PartName="/ppt/slideLayouts/slideLayout1.xml", ContentType="application/vnd.openxmlformats-officedocument.presentationml.slideLayout+xml")
    etree.SubElement(content_types, Q('ct','Override'), PartName="/ppt/slideMasters/slideMaster1.xml", ContentType="application/vnd.openxmlformats-officedocument.presentationml.slideMaster+xml")
    etree.SubElement(content_types, Q('ct','Override'), PartName="/ppt/theme/theme1.xml", ContentType="application/vnd.openxmlformats-officedocument.theme+xml")
    root_rels = etree.Element(Q('rel','Relationships'))
    etree.SubElement(root_rels, Q('rel','Relationship'), Id="rId1", Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument", Target="ppt/presentation.xml")
    pres = etree.Element(Q('p','presentation'), nsmap={'a': NS['a'], 'r': NS['r'], 'p': NS['p']})
    pres.set('saveSubsetFonts', '1')
    sldMasterIdLst = etree.SubElement(pres, Q('p','sldMasterIdLst'))
    etree.SubElement(sldMasterIdLst, Q('p','sldMasterId'), id="2147483648", **{Q('r','id'): 'rId1'})
    sldIdLst = etree.SubElement(pres, Q('p','sldIdLst'))
    for i in range(n):
        etree.SubElement(sldIdLst, Q('p','sldId'), id=str(256+i), **{Q('r','id'): f'rId{i+10}'})
    etree.SubElement(pres, Q('p','sldSz'), cx=SW, cy=SH, type="custom")
    etree.SubElement(pres, Q('p','notesSz'), cx=SH, cy=SW)
    pres_rels = etree.Element(Q('rel','Relationships'))
    etree.SubElement(pres_rels, Q('rel','Relationship'), Id="rId1", Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/slideMaster", Target="slideMasters/slideMaster1.xml")
    for i in range(n):
        etree.SubElement(pres_rels, Q('rel','Relationship'), Id=f"rId{i+10}", Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/slide", Target=f"slides/slide{i+1}.xml")
    etree.SubElement(pres_rels, Q('rel','Relationship'), Id="rId100", Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/theme", Target="theme/theme1.xml")
    theme = etree.Element(Q('a','theme'), nsmap={'a': NS['a']}, name="CoCoDuo")
    themeElements = etree.SubElement(theme, Q('a','themeElements'))
    clrScheme = etree.SubElement(themeElements, Q('a','clrScheme'), name="CoCoDuo")
    for tag, val in [('dk1',DARK),('lt1',WHITE),('dk2',DARK),('lt2',CREAM),('accent1',TEAL),('accent2',ORANGE),('accent3',GREEN),('accent4',BLUE),('accent5',YELLOW),('accent6',RED),('hlink',TEAL),('folHlink',MD_GRAY)]:
        el = etree.SubElement(clrScheme, Q('a',tag))
        etree.SubElement(el, Q('a','srgbClr'), val=val)
    fontScheme = etree.SubElement(themeElements, Q('a','fontScheme'), name="CoCoDuo")
    for ft in ['majorFont','minorFont']:
        font = etree.SubElement(fontScheme, Q('a',ft))
        etree.SubElement(font, Q('a','latin'), typeface="Calibri")
        etree.SubElement(font, Q('a','ea'), typeface="")
        etree.SubElement(font, Q('a','cs'), typeface="")
    fmtScheme = etree.SubElement(themeElements, Q('a','fmtScheme'), name="CoCoDuo")
    for lst_name in ['fillStyleLst','lnStyleLst','effectStyleLst','bgFillStyleLst']:
        lst = etree.SubElement(fmtScheme, Q('a',lst_name))
        sf = etree.SubElement(lst, Q('a','solidFill'))
        etree.SubElement(sf, Q('a','schemeClr'), val='phClr')
    etree.SubElement(theme, Q('a','objectDefaults'))
    etree.SubElement(theme, Q('a','extraClrSchemeLst'))
    slide_master = etree.Element(Q('p','sldMaster'), nsmap={'a': NS['a'], 'r': NS['r'], 'p': NS['p']})
    cSld_m = etree.SubElement(slide_master, Q('p','cSld'))
    bg_m = etree.SubElement(cSld_m, Q('p','bg'))
    bgPr_m = etree.SubElement(bg_m, Q('p','bgPr'))
    sf_m = etree.SubElement(bgPr_m, Q('a','solidFill'))
    etree.SubElement(sf_m, Q('a','srgbClr'), val=CREAM)
    etree.SubElement(bgPr_m, Q('a','effectLst'))
    spTree_m = etree.SubElement(cSld_m, Q('p','spTree'))
    nvGrp_m = etree.SubElement(spTree_m, Q('p','nvGrpSpPr'))
    etree.SubElement(nvGrp_m, Q('p','cNvPr'), id="1", name="")
    etree.SubElement(nvGrp_m, Q('p','cNvGrpSpPr'))
    etree.SubElement(nvGrp_m, Q('p','nvPr'))
    grpPr_m = etree.SubElement(spTree_m, Q('p','grpSpPr'))
    xfrm_m = etree.SubElement(grpPr_m, Q('a','xfrm'))
    etree.SubElement(xfrm_m, Q('a','off'), x="0", y="0")
    etree.SubElement(xfrm_m, Q('a','ext'), cx=SW, cy=SH)
    etree.SubElement(xfrm_m, Q('a','chOff'), x="0", y="0")
    etree.SubElement(xfrm_m, Q('a','chExt'), cx=SW, cy=SH)
    sldLayoutIdLst = etree.SubElement(slide_master, Q('p','sldLayoutIdLst'))
    etree.SubElement(sldLayoutIdLst, Q('p','sldLayoutId'), id="2147483649", **{Q('r','id'): 'rId1'})
    sm_rels = etree.Element(Q('rel','Relationships'))
    etree.SubElement(sm_rels, Q('rel','Relationship'), Id="rId1", Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/slideLayout", Target="../slideLayouts/slideLayout1.xml")
    etree.SubElement(sm_rels, Q('rel','Relationship'), Id="rId2", Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/theme", Target="../theme/theme1.xml")
    slide_layout = etree.Element(Q('p','sldLayout'), nsmap={'a': NS['a'], 'r': NS['r'], 'p': NS['p']}, type="blank", preserve="1")
    cSld_l = etree.SubElement(slide_layout, Q('p','cSld'), name="Blank")
    spTree_l = etree.SubElement(cSld_l, Q('p','spTree'))
    nvGrp_l = etree.SubElement(spTree_l, Q('p','nvGrpSpPr'))
    etree.SubElement(nvGrp_l, Q('p','cNvPr'), id="1", name="")
    etree.SubElement(nvGrp_l, Q('p','cNvGrpSpPr'))
    etree.SubElement(nvGrp_l, Q('p','nvPr'))
    etree.SubElement(spTree_l, Q('p','grpSpPr'))
    sl_rels = etree.Element(Q('rel','Relationships'))
    etree.SubElement(sl_rels, Q('rel','Relationship'), Id="rId1", Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/slideMaster", Target="../slideMasters/slideMaster1.xml")
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as z:
        z.writestr('[Content_Types].xml', decl + etree.tostring(content_types, xml_declaration=False))
        z.writestr('_rels/.rels', decl + etree.tostring(root_rels, xml_declaration=False))
        z.writestr('ppt/presentation.xml', decl + etree.tostring(pres, xml_declaration=False))
        z.writestr('ppt/_rels/presentation.xml.rels', decl + etree.tostring(pres_rels, xml_declaration=False))
        z.writestr('ppt/theme/theme1.xml', decl + etree.tostring(theme, xml_declaration=False))
        z.writestr('ppt/slideMasters/slideMaster1.xml', decl + etree.tostring(slide_master, xml_declaration=False))
        z.writestr('ppt/slideMasters/_rels/slideMaster1.xml.rels', decl + etree.tostring(sm_rels, xml_declaration=False))
        z.writestr('ppt/slideLayouts/slideLayout1.xml', decl + etree.tostring(slide_layout, xml_declaration=False))
        z.writestr('ppt/slideLayouts/_rels/slideLayout1.xml.rels', decl + etree.tostring(sl_rels, xml_declaration=False))
        for i, cSld_elem in enumerate(slides):
            sld = etree.Element(Q('p','sld'), nsmap={'a': NS['a'], 'r': NS['r'], 'p': NS['p']})
            sld.append(cSld_elem)
            sld_rels = etree.Element(Q('rel','Relationships'))
            etree.SubElement(sld_rels, Q('rel','Relationship'), Id="rId1", Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/slideLayout", Target="../slideLayouts/slideLayout1.xml")
            z.writestr(f'ppt/slides/slide{i+1}.xml', decl + etree.tostring(sld, xml_declaration=False))
            z.writestr(f'ppt/slides/_rels/slide{i+1}.xml.rels', decl + etree.tostring(sld_rels, xml_declaration=False))
    return output_path

output = make_pptx(slides_data, '/tmp/CoCoDuo_Migration_Launchpad.pptx')
fsize = os.path.getsize(output)
print(f"\nPPTX saved: {output}")
print(f"File size: {fsize/1024:.1f} KB")
print(f"Slides: {len(slides_data)}")
print(f"Theme: Light (cream/warm gray background)")
print("\nSlide Outline:")
for i,t in enumerate([
    "Title \u2014 CoCoDuo Migration Launchpad",
    "Problem Statement \u2014 Migration Planning Is Broken",
    "CoCo Solution \u2014 AI-Powered Migration Launchpad",
    "Old Way vs. CoCo Way \u2014 Side-by-Side Comparison",
    "Time Savings \u2014 From Weeks to Minutes",
    "Expected Outcomes & Measurable Impact",
    "Technology Stack & Architecture",
    "Team Roles & Responsibilities",
    "Assumptions, Risks & Trial Account Workarounds",
    "Closing \u2014 Built in 6 Hours, Saves Weeks, Scales Infinitely",
], 1):
    print(f"  {i}. {t}")

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("PUT 'file:///tmp/CoCoDuo_Migration_Launchpad.pptx' @MIGRATION_ASSESSMENTS.INTAKE.MIGRATION_PLANS AUTO_COMPRESS=FALSE OVERWRITE=TRUE").collect()
result = session.sql("SELECT GET_PRESIGNED_URL(@MIGRATION_ASSESSMENTS.INTAKE.MIGRATION_PLANS, 'CoCoDuo_Migration_Launchpad.pptx', 3600)").collect()
print("Download URL (valid 1 hour):")
print(result[0][0])